In [1]:

import cv2
import mediapipe as mp
import numpy as np
import time


In [2]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = "face_landmarker.task"

base_options = python.BaseOptions(model_asset_path=model_path)

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_faces=1
)

landmarker = vision.FaceLandmarker.create_from_options(options)


In [3]:

def difrence(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))


In [4]:

def calculate_EAR(eye_points):
    p1, p2, p3, p4, p5, p6 = eye_points

    vertical1 = difrence(p2, p6)
    vertical2 = difrence(p3, p5)
    horizontal = difrence(p1, p4)

    ear = (vertical1 + vertical2) / (2.0 * horizontal)
    return ear


In [5]:

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

cap = cv2.VideoCapture(0)

frame_timestamp_ms = 0
closed_frames = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)

    h, w, channels = frame.shape

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=image_rgb
    )

    results = landmarker.detect_for_video(mp_image, frame_timestamp_ms)
    frame_timestamp_ms += 33

    status = "No Face"
    color = (255, 255, 255) 

    if results.face_landmarks:

        for face in results.face_landmarks:

            # ===== استخراج نقاط العين =====
            left_eye = []
            right_eye = []

            for idx in LEFT_EYE:
                lm = face[idx]
                left_eye.append((lm.x * w, lm.y * h))

            for idx in RIGHT_EYE:
                lm = face[idx]
                right_eye.append((lm.x * w, lm.y * h))

            left_ear = calculate_EAR(left_eye)
            right_ear = calculate_EAR(right_eye)
            ear = (left_ear + right_ear) / 2.0

            for (x, y) in left_eye + right_eye:
                cv2.circle(frame, (int(x), int(y)), 2, (0,255,0), -1)

            if ear < 0.18:
                closed_frames += 1
            else:
                closed_frames = 0

            if closed_frames > 15:
                status = "Drive is Sleep!!!"
                color = (0, 0, 255)
            else:
                status = "Drive is Awake"
                color = (0, 255, 0)

            # ===== عرض EAR =====
            cv2.putText(frame, f"EAR: {ear:.2f}", (30, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    # ===== عرض الحالة =====
    cv2.putText(frame, status, (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)

    cv2.imshow("Driver Monitor", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()